# Lab 2 — Make every reply safe to send

**~25 minutes · nothing to fill in · Run All takes about 2 minutes**

In Lab 1 the desk wrote a reply for every ticket. Each reply goes to a real bank customer. Some
mistakes in a reply cost the bank much more than a slow answer:

- **promising a refund** that the owning team has not approved,
- **giving a deadline** that nobody on the team agreed to, and
- **asking for an OTP, PIN or password**, which is exactly what fraudsters do.

The queue now has a hard ticket too. `GB-T-4475` demands a refund **by tomorrow**. A writer who wants
to please the customer is likely to promise it.

In this lab you give the crew a **guardrail**: a check that runs on the writer's answer. If the reply
breaks a rule, CrewAI sends the reason back and the writer tries again. No unsafe reply leaves the desk.

**Words used in this lab**

- **Guardrail:** a check that runs on a task's output. If it fails, the agent gets the reason and tries
  again.
- **Retry:** one more attempt at the same task. `guardrail_max_retries` sets how many are allowed.

## 1 · Setup: the desk from Lab 1

This cell rebuilds the Lab 1 desk as a function, so you can build it with different settings. There are
two settings:

- `writer_backstory`: the writer's instructions.
- `guardrail`: a check for the writer's task, or `None` for no check.

In [ ]:
import os, time
from crewai import LLM, Agent, Task, Crew, Process
from desk_kit import TICKETS, TEAM_RULE, check_reply, print_queue, score

llm = LLM(model=os.environ["OPENAI_MODEL"], base_url=os.environ["OPENAI_BASE_URL"],
          api_key=os.environ["OPENAI_API_KEY"], temperature=0)

from crewai.tools import tool

@tool("ticket_lookup")
def ticket_lookup(ticket_id: str) -> str:
    """Return the text of a Global Bank customer support ticket by its id, e.g. GB-T-4471."""
    return TICKETS.get(ticket_id, f"No ticket found with id {ticket_id}")

from pydantic import BaseModel, Field

class Triage(BaseModel):
    category: str = Field(description="The kind of problem, in a few words")
    team: str = Field(description="The team that owns the ticket")
    priority: str = Field(description="High, Medium or Low")

class DeskReply(BaseModel):
    customer_reply: str = Field(description="The reply the customer reads, at most four sentences")
    summary: str = Field(description="Handover: what happened, in one sentence")
    what_to_check: str = Field(description="Handover: what the owning team should check first")
    next_action: str = Field(description="Handover: the next thing the owning team should do")


def build_desk(writer_backstory, guardrail=None):
    researcher = Agent(role="Ticket Researcher",
                       goal="Retrieve the ticket and state the facts in it, without interpreting them",
                       backstory="You pull the raw ticket and never guess beyond what it says.",
                       llm=llm, tools=[ticket_lookup], allow_delegation=False)
    classifier = Agent(role="Support Triage Analyst",
                       goal="Classify a ticket, name the owning team and set a priority",
                       backstory="You triage the Global Bank customer support desk.",
                       llm=llm, allow_delegation=False)
    writer = Agent(role="Response Drafter",
                   goal="Write the first reply the bank customer will read, and the handover note",
                   backstory=writer_backstory, llm=llm, allow_delegation=False)
    t1 = Task(description="Retrieve ticket {ticket_id} and list the facts it contains.",
              expected_output="A short bulleted list of facts, no interpretation.", agent=researcher)
    t2 = Task(description=f"Classify ticket {{ticket_id}} and name the owning team. {TEAM_RULE} "
                          "The priority is High if the customer has lost money, otherwise Medium or Low.",
              expected_output="The category, the team and the priority.", agent=classifier,
              output_pydantic=Triage)
    t3 = Task(description="Write the first reply to the customer who sent ticket {ticket_id}, "
                          "and a handover note for the owning team.",
              expected_output="The customer reply and the three parts of the handover note.",
              agent=writer, output_pydantic=DeskReply,
              guardrail=guardrail, guardrail_max_retries=2)
    return Crew(agents=[researcher, classifier, writer], tasks=[t1, t2, t3],
                process=Process.sequential, verbose=False)

async def run_queue(crew):
    rows = []
    for ticket_id in TICKETS:
        out = await crew.kickoff_async(inputs={"ticket_id": ticket_id})
        decision, reply = crew.tasks[1].output.pydantic, out.pydantic
        rows.append({"ticket": ticket_id, "team": decision.team, "priority": decision.priority,
                     "reply": reply.customer_reply, "summary": reply.summary,
                     "what_to_check": reply.what_to_check, "next_action": reply.next_action})
    return rows

print("desk ready")

## 2 · The bank's rules, as code

`check_reply` in `desk_kit.py` holds the three rules. It takes a reply and returns the list of rules the
reply breaks. An empty list means the reply is safe to send.

It is plain Python, not a model. So it is fast, it costs nothing, and it gives the same answer every
time. Try it on a few sentences:

In [ ]:
samples = [
    "We will refund the Rs 3,000 by tomorrow.",
    "Please share the OTP you received so we can check your login.",
    "Never share your OTP with anyone, including bank staff.",
    "Our Transactions team is reviewing both payments and will write to you.",
]
for s in samples:
    print(check_reply(s) or "safe", "  <-", s)

Read the source if you want to see how each rule is checked:

```python
import inspect, desk_kit
print(inspect.getsource(desk_kit.check_reply))
```

A check like this can only catch what you can describe exactly. It cannot tell whether a reply is
kind, or correct. It catches the few mistakes that must never reach a customer, every time.

## 3 · A writer who wants to please

Here the writer's instructions ask it to make every customer feel reassured, and say nothing about the
bank's rules. Teams write instructions like this all the time.

Run the queue with **no guardrail**, and check every reply.

In [ ]:
EAGER = "You write to Global Bank customers. You want every customer to feel fully reassured and satisfied."

eager_queue = await run_queue(build_desk(EAGER))

print_queue(eager_queue)
print()
for row in eager_queue:
    problems = check_reply(row["reply"])
    if problems:
        print(row["ticket"], "-", row["reply"])
        for p in problems:
            print("    BROKEN:", p)
        print()

**Would you send these?** Look at `GB-T-4475` in particular. The customer asked for a refund by
tomorrow. A reply that agrees has now made a promise in the bank's name.

If your run shows no broken rules, run the cell again. A model does not make the same mistake every
time. That is the reason for a check that runs every time.

## 4 · Add the guardrail

A CrewAI guardrail is a function. It gets the task's output and returns a pair:

- `(True, output)`: the output is fine, and it goes on.
- `(False, "reason")`: CrewAI sends the reason back to the writer, and the writer tries again.

`guardrail_max_retries=2` allows two more attempts. If every attempt fails, the task fails with an
error. A human then picks up that ticket. That is the right result: no unsafe reply goes out.

The writer's instructions stay the same as in section 3. Only the guardrail is new.

In [ ]:
CAUGHT = []

def reply_is_safe(output):
    problems = check_reply(output.pydantic.customer_reply)
    if problems:
        CAUGHT.append(problems)
        return (False, " ".join(problems) + " Rewrite the customer reply so it keeps these rules.")
    return (True, output)

safe_queue = await run_queue(build_desk(EAGER, guardrail=reply_is_safe))

print_queue(safe_queue)
print()
print("no guardrail  :", score(eager_queue))
print("with guardrail:", score(safe_queue))
print(f"the guardrail sent {len(CAUGHT)} replies back for a rewrite")

Now read the replies that were sent back, as they finally came out:

In [ ]:
for before, after in zip(eager_queue, safe_queue):
    if check_reply(before["reply"]):
        print(after["ticket"])
        print("  before:", before["reply"])
        print("  after :", after["reply"])
        print()

### What to take away

- **The guardrail made the `safe reply` column complete** without changing the writer at all. The
  rules live in one place, in code, and they run on every ticket.
- **State the rules in the prompt too.** The Lab 1 writer was told "promise only what the team can do,
  and never invent a timeline", so it broke fewer rules to begin with. The prompt asks, and the
  guardrail checks. Each rewrite is one more model request, so fewer broken rules also means a
  faster desk.
- **A guardrail cannot add what the writer does not know.** Look at the `says what happened` column.
  It did not change. The desk still reads only the ticket. Lab 3 fixes that.

### Failure modes to remember

- **Delegation in circles.** With `allow_delegation=True` on several agents, they can pass work to each
  other in circles. Turn it on for one agent at a time, and only on purpose.
- **A loose `expected_output`.** If a task does not say what it must produce, the next task guesses.
  `output_pydantic` is the strictest form of `expected_output`.
- **A check with no limit.** A guardrail that can never pass would retry forever.
  `guardrail_max_retries` is the limit. Always set it.

---

**Next:** Lab 3 builds the desk again in Google ADK, and gives it the bank's records.